In [20]:
import pandas as pd
import numpy as np

In [21]:
cc_calls = pd.read_csv(
    "../../data/01_raw/raw_cc_calls.csv",
    low_memory=False
)

print(f"Shape: {cc_calls.shape}")
cc_calls.shape

Shape: (32882, 33)


(32882, 33)

In [22]:
cc_calls.info()
cc_calls.head()

<class 'pandas.DataFrame'>
RangeIndex: 32882 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Contact_ID                                32882 non-null  float64
 1   Call_Date                                 32882 non-null  str    
 2   Direction                                 32882 non-null  str    
 3   cc_care_package                           32744 non-null  str    
 4   cc_care_package_discussed                 32744 non-null  str    
 5   cc_urgency_getting_on_site                32744 non-null  str    
 6   cc_external_consultant                    32744 non-null  str    
 7   cc_agent_cross_sell_attempt               32744 non-null  str    
 8   cc_customer_issues_concerns               32744 non-null  str    
 9   cc_business_struggles_financial_hardship  32744 non-null  str    
 10  cc_call_initiated_by                      327

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,6.255130e+11,08-05-2025,OUT_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,30,20,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,5.910870e+11,25-11-2024,OUT_BOUND,Standard,Yes,No,No,No,Yes,No,...,0,0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,5.650910e+11,23-10-2024,IN_BOUND,Standard,Yes,No,No,No,Yes,No,...,40,20,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,5.939750e+11,13-01-2025,IN_BOUND,Premier,Yes,No,No,No,Yes,Yes,...,40,30,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,6.222820e+11,19-03-2025,IN_BOUND,Standard,Yes,No,No,No,Yes,Yes,...,40,20,Yes,Yes,No,Yes,Yes,ED6707,1,2025


In [23]:
cc_calls = cc_calls.drop_duplicates()
print("Shape after duplicate removal:", cc_calls.shape)

Shape after duplicate removal: (32789, 33)


In [24]:
cc_calls.columns = (
    cc_calls.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [25]:
for col in cc_calls.columns:
    if "date" in col:
        cc_calls[col] = pd.to_datetime(
            cc_calls[col], errors="coerce"
        )

In [26]:
missing_df = (
    cc_calls.isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_df.head(20)

call_date                                   58.537924
co_ref                                       3.516423
cc_issues_within_questionnaire               1.421208
cc_call_initiated_by                         0.417823
cc_care_package_discussed                    0.417823
cc_agent_cross_sell_attempt                  0.417823
cc_external_consultant                       0.417823
cc_customer_issues_concerns                  0.417823
cc_care_package                              0.417823
cc_urgency_getting_on_site                   0.417823
cc_business_struggles_financial_hardship     0.417823
cc_contractor_suggest_leave                  0.289731
cc_refund_discussed                          0.289731
cc_contractor_sentiment_overall_score        0.289731
cc_contractor_sentiment_end_score            0.289731
cc_contractor_sentiment                      0.289731
cc_contractor_sentiment_issues_score         0.289731
cc_pricing_mentioned                         0.289731
cc_contractor_complained    

In [27]:
cat_cols = cc_calls.select_dtypes(include="object").columns.tolist()
num_cols = cc_calls.select_dtypes(include=np.number).columns.tolist()

# protect id columns
id_cols = [col for col in num_cols if "id" in col or "ref" in col]
num_cols = [col for col in num_cols if col not in id_cols]

print("Categorical:", len(cat_cols))
print("Numeric:", len(num_cols))

Categorical: 29
Numeric: 2


C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_43348\352053162.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = cc_calls.select_dtypes(include="object").columns.tolist()


In [28]:
# -------------------------------
# categorical columns
# -------------------------------
for col in cat_cols:
    if cc_calls[col].isnull().sum() > 0:
        cc_calls[col] = cc_calls[col].fillna("No_CC_Interaction")

# -------------------------------
# numeric columns
# -------------------------------
for col in num_cols:
    if cc_calls[col].isnull().sum() > 0:
        unique_vals = cc_calls[col].dropna().nunique()

        # binary columns
        if unique_vals <= 2:
            cc_calls[col] = cc_calls[col].fillna(0)

        # count style
        elif any(word in col.lower() for word in ["count", "num", "calls", "attempts"]):
            cc_calls[col] = cc_calls[col].fillna(0)

        # continuous metrics
        else:
            cc_calls[col] = cc_calls[col].fillna(
                cc_calls[col].median()
            )

In [29]:
cc_calls.dtypes

contact_id                                         float64
call_date                                   datetime64[us]
direction                                              str
cc_care_package                                        str
cc_care_package_discussed                              str
cc_urgency_getting_on_site                             str
cc_external_consultant                                 str
cc_agent_cross_sell_attempt                            str
cc_customer_issues_concerns                            str
cc_business_struggles_financial_hardship               str
cc_call_initiated_by                                   str
cc_questionnaire_completion                            str
cc_chasing_response                                    str
cc_issues_within_questionnaire                         str
cc_login_issues                                        str
cc_platform_issues                                     str
cc_dissatisfaction_time_to_complete                    s

In [30]:
cc_calls.to_csv(
    "../../data/02_processed/processed_cc_calls.csv",
    index=False
)